In [102]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [103]:
billing_df=pd.read_csv('Billing_Data.csv')

### 1.	Load the patient dataset and show summary with info().

In [104]:
patient_df=pd.read_csv('Patient_Data .csv')
patient_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


### 2.	Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].

In [105]:
billing_cols_df = patient_df[['PatientID', 'Department', 'Doctor', 'BillAmount']]
billing_cols_df

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN
5,101,Cardiology,Dr. Smith,5000.0


### 3.	Drop administrative columns like ['ReceptionistID', 'CheckInTime'].

In [106]:
patient_df = patient_df.drop(columns=['ReceptionistID', 'CheckInTime'])
patient_df

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN
5,101,Alice,Cardiology,Dr. Smith,5000.0


### 4.	Use groupby to find total bill amount per department.

In [107]:
total_bill = patient_df.groupby('Department')['BillAmount'].sum()
print(total_bill)

Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64


### 5.	Remove duplicate patient records based on PatientID.

In [108]:
patient_df = patient_df.drop_duplicates(subset='PatientID')
patient_df

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN


### 6.	Fill missing BillAmount values with the mean bill amount.

In [109]:
mean_bill = patient_df['BillAmount'].mean()
patient_df['BillAmount'] = patient_df['BillAmount'].fillna(mean_bill)
patient_df

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000
1,102,Bob,Neurology,Dr. John,6233.333333
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000
3,104,David,Cardiology,Dr. Smith,6200.000000
4,105,Eva,Dermatology,Dr. Rose,6233.333333


### 7.	Merge the billing dataset with patient dataset on PatientID.

In [110]:
merged_df = pd.merge(patient_df, billing_df, on='PatientID', how='inner')
merged_df

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Bob,Neurology,Dr. John,6233.333333,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000,4000


### 8.	Concatenate an additional DataFrame that contains new patients for the current week (row-wise).

In [111]:
new_patients = pd.DataFrame({
    'PatientID': [201, 202],
    'Name':['mama','texx'],
    'Department': ['Cardiology', 'Neurology'],
    'Doctor': ['Dr. A', 'Dr. B'],
    'BillAmount': [8000, 6000]
})

In [112]:
patient_df = pd.concat([patient_df, new_patients], axis=0, ignore_index=True)
patient_df

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000
1,102,Bob,Neurology,Dr. John,6233.333333
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000
3,104,David,Cardiology,Dr. Smith,6200.000000
4,105,Eva,Dermatology,Dr. Rose,6233.333333
5,201,mama,Cardiology,Dr. A,8000.000000
6,202,texx,Neurology,Dr. B,6000.000000


### 9.	Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).

In [113]:
new_cols = pd.DataFrame({
    'InsuranceCovered': [True] * len(patient_df),
    'FinalAmount': patient_df['BillAmount'] * 0.9
})

In [114]:
patient_df = pd.concat([patient_df, new_cols], axis=1)
patient_df

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,True,4500.0
1,102,Bob,Neurology,Dr. John,6233.333333,True,5610.0
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,True,6750.0
3,104,David,Cardiology,Dr. Smith,6200.000000,True,5580.0
4,105,Eva,Dermatology,Dr. Rose,6233.333333,True,5610.0
5,201,mama,Cardiology,Dr. A,8000.000000,True,7200.0
6,202,texx,Neurology,Dr. B,6000.000000,True,5400.0


#### This produces your final cleaned dataset with:

#### duplicates removed
#### missing values handled
#### datasets merged
#### new rows and columns added

#### You can now directly use "patient_df" for department revenue analysis or doctor performance.